In [1]:
import os
import psycopg2
import numpy as np
from dotenv import load_dotenv
from google import genai
from pgvector.psycopg2 import register_vector

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="pillwise",
    user="pillwise",
    password="pillwise123"
)
register_vector(conn)

print("ready")

ready


In [3]:
def retrieve(query, top_k=5):

    result = client.models.embed_content(
        model = "gemini-embedding-001",
        contents = query
    )
    query_embedding = result.embeddings[0].values

    cur = conn.cursor()
    cur.execute("""
        SELECT drug, chunk_text, 1 - (embedding <=> %s::vector) AS similarity
        FROM chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (query_embedding, query_embedding, top_k))

    results = cur.fetchall()
    return results

results = retrieve("what is the dosage for acetaminophen?")
for i, row in enumerate(results):
    print(f"\n --- result {i+1} --")
    print(f"drug : {row[0]}")
    print(f"similarity : {row[2]: .4f}")
    print(f"chunk : {row[1][:200]}")


 --- result 1 --
drug : ibuprofen
similarity :  0.6846
chunk : y two weeks.
After a satisfactory response has been achieved, the patient's dose should be reviewed
and adjusted as required.
Mild to moderate pain:400 mg every 4 to 6 hours as necessary for relief of

 --- result 2 --
drug : ibuprofen
similarity :  0.6833
chunk : 2 years and over: take 1 tablet every 4 to 6 hours while
symptoms persist
if pain or fever does not respond to 1 tablet, 2 tablets may be used
do not exceed 6 tablets in 24 hours, unless directed by a

 --- result 3 --
drug : acetaminophen
similarity :  0.6784
chunk : duct contains acetaminophen. Severe liver damage may occur
if you take
•
•
•
Allergy alert: Acetaminophen may cause severe skin reactions. Symptoms may
include:
•
•
•
cough due to minor throat and bro

 --- result 4 --
drug : ibuprofen
similarity :  0.6750
chunk : uency should be adjusted to suit an individual patient's needs.
Do not exceed 3200 mg total daily dose. If gastrointestinal complaints oc

In [7]:
import requests as req
import json

def ask(query, top_k=5):
    #retrieve chunks
    results = retrieve(query, top_k)

    #builf context from chunks
    context = ""
    for i, row in enumerate(results):
        context += f"[{i+1}] ({row[0]}) {row[1]}\n\n"

    #build prompt
    prompt = f"""You are a medical information assistant. Answer the question using ONLY the context provided below.
For each fact you state, cite the source number in square brackets like [1], [2].
If the context doesn't contain enough information, say so.

Context:
{context}

Question: {query}

Answer:"""
    #call ollama
    response = req.post("http://localhost:11434/api/generate", json={
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False
    })

    lines = response.text.strip().split("\n")
    answer = json.loads(lines[0])["response"]
    return answer, results

#test
answer, sources = ask("what is the dosage for acetaminophen?")
print(answer)
    

The context only provides information on ibuprofen and does not mention acetaminophen directly. However, [3] (acetaminophen) mentions that excessive doses of 4000 mg or more in a 24-hour period may cause severe liver damage.

Unfortunately, the dosage for acetaminophen is not explicitly stated in the provided context. The dosage information is only available for ibuprofen.


In [8]:
answer, sources = ask("what are the warnings for ibuprofen?")
print(answer)

The warnings for ibuprofen include:

* Cardiovascular Thrombotic Events: an increased risk of serious cardiovascular events, including myocardial infarction (MI) and stroke, which can be fatal [3]
* Gastrointestinal Risk: NSAIDs cause an increased risk of serious gastrointestinal adverse events, including bleeding, ulceration, and perforation of the stomach or intestines, which can be fatal. Elderly patients are at greater risk for serious gastrointestinal events [2]
* Anaphylactoid Reactions: ibuprofen tablets are contraindicated in patients with severe heart failure unless benefits outweigh risks of worsening heart failure; monitor for signs of worsening heart failure [5]

Note that the context does not provide a comprehensive list of warnings, but rather mentions specific warnings and cautions related to cardiovascular events, gastrointestinal risk, and heart failure.


In [9]:
from IPython.display import Markdown, display
display(Markdown(answer))

The warnings for ibuprofen include:

* Cardiovascular Thrombotic Events: an increased risk of serious cardiovascular events, including myocardial infarction (MI) and stroke, which can be fatal [3]
* Gastrointestinal Risk: NSAIDs cause an increased risk of serious gastrointestinal adverse events, including bleeding, ulceration, and perforation of the stomach or intestines, which can be fatal. Elderly patients are at greater risk for serious gastrointestinal events [2]
* Anaphylactoid Reactions: ibuprofen tablets are contraindicated in patients with severe heart failure unless benefits outweigh risks of worsening heart failure; monitor for signs of worsening heart failure [5]

Note that the context does not provide a comprehensive list of warnings, but rather mentions specific warnings and cautions related to cardiovascular events, gastrointestinal risk, and heart failure.

In [10]:
from IPython.display import Markdown, display

answer, sources = ask("what are the warnings for ibuprofen?")
display(Markdown(answer))

According to the provided context:

There are several warnings for ibuprofen:

1. Cardiovascular Thrombotic Events:
   * Increased risk of serious cardiovascular (CV) thrombotic events, including myocardial infarction (MI) and stroke, which can be fatal. [3]
2. Gastrointestinal Risk:
   * Increased risk of serious gastrointestinal adverse events, including bleeding, ulceration, and perforation of the stomach or intestines, which can be fatal. These events can occur at any time during use without warning symptoms. Elderly patients are at greater risk for serious gastrointestinal events. [2]
3. Severe Heart Failure:
   * Avoid using ibuprofen in patients with severe heart failure unless the benefits are expected to outweigh the risk of worsening heart failure. Monitor patients for signs of worsening heart failure. [5]

Please note that these warnings are specific to the context provided and may not be an exhaustive list of all possible warnings for ibuprofen.